In [1]:
import pandas as pd

# Day 4 — Step 1.1
# Synthetic Security Master (not real market data)

security_master = pd.DataFrame([
    {
        "security_id": 1001,
        "ticker": "ALPHA",
        "listing_date": "2010-01-04",
        "delisting_date": None,
        "exchange": "NYSE",
        "share_type": "COMMON",
    },
    {
        "security_id": 1002,
        "ticker": "BETA",
        "listing_date": "2011-06-01",
        "delisting_date": "2018-09-28",
        "exchange": "NASDAQ",
        "share_type": "COMMON",
    },
    {
        "security_id": 1003,
        "ticker": "GAMMA",
        "listing_date": "2020-04-15",
        "delisting_date": None,
        "exchange": "NASDAQ",
        "share_type": "COMMON",
    },
    {
        "security_id": 1004,
        "ticker": "DELTA",
        "listing_date": "2012-03-01",
        "delisting_date": None,
        "exchange": "NYSE",
        "share_type": "ETF",
    },
    {
        "security_id": 1005,
        "ticker": "EPSILON",
        "listing_date": "2013-01-02",
        "delisting_date": None,
        "exchange": "OTC",
        "share_type": "COMMON",
    },
])

for column in ["listing_date", "delisting_date"]:
    security_master[column] = pd.to_datetime(
        security_master[column]
    )

display(security_master)

print("Shape:", security_master.shape)
print(
    "Duplicate security IDs:",
    security_master["security_id"].duplicated().sum()
)

,security_id,ticker,listing_date,delisting_date,exchange,share_type
0,1001,ALPHA,2010-01-04,NaT,NYSE,COMMON
1,1002,BETA,2011-06-01,2018-09-28,NASDAQ,COMMON
2,1003,GAMMA,2020-04-15,NaT,NASDAQ,COMMON
3,1004,DELTA,2012-03-01,NaT,NYSE,ETF
4,1005,EPSILON,2013-01-02,NaT,OTC,COMMON


Shape: (5, 6)
Duplicate security IDs: 0


In [2]:
# Day 4 — Step 2.1
# Point-in-Time Universe Eligibility Prototype

def get_pit_universe(security_master, as_of_date):
    as_of_date = pd.Timestamp(as_of_date)

    # 1. Already listed by the selection date
    listed = (
        security_master["listing_date"] <= as_of_date
    )

    # 2. Not delisted before the selection date
    active = (
        security_master["delisting_date"].isna()
        | (security_master["delisting_date"] >= as_of_date)
    )

    # 3. Eligible US exchanges
    valid_exchange = security_master["exchange"].isin(
        ["NYSE", "NASDAQ", "NYSE AMERICAN"]
    )

    # 4. Common shares only
    valid_share_type = (
        security_master["share_type"] == "COMMON"
    )

    eligible = (
        listed
        & active
        & valid_exchange
        & valid_share_type
    )

    return (
        security_master.loc[
            eligible,
            ["security_id", "ticker",
             "listing_date", "delisting_date"]
        ]
        .sort_values("security_id")
        .reset_index(drop=True)
    )


# Test historical universe construction
for date in ["2015-06-30", "2019-06-30", "2021-06-30"]:
    universe = get_pit_universe(
        security_master, date
    )

    print(f"\nUniverse as of {date}:")
    print(universe["ticker"].tolist())


Universe as of 2015-06-30:
['ALPHA', 'BETA']

Universe as of 2019-06-30:
['ALPHA']

Universe as of 2021-06-30:
['ALPHA', 'GAMMA']


In [3]:
# Day 4 — Step 2.2
# Boundary-date checks

test_dates = [
    "2011-05-31",  # Before BETA's listing
    "2011-06-01",  # BETA listing date
    "2018-09-28",  # BETA delisting date
    "2018-10-01",  # After BETA's delisting
    "2020-04-15",  # GAMMA listing date
]

for date in test_dates:
    universe = get_pit_universe(
        security_master, date
    )
    print(
        date,
        "->",
        universe["ticker"].tolist()
    )


2011-05-31 -> ['ALPHA']
2011-06-01 -> ['ALPHA', 'BETA']
2018-09-28 -> ['ALPHA', 'BETA']
2018-10-01 -> ['ALPHA']
2020-04-15 -> ['ALPHA', 'GAMMA']


In [4]:
# Day 4 — Step 3.1
# Automated PIT Universe Tests

def get_tickers(as_of_date):
    universe = get_pit_universe(
        security_master,
        as_of_date
    )
    return set(universe["ticker"])


# Test 1: Historical eligibility
assert get_tickers("2015-06-30") == {
    "ALPHA", "BETA"
}

# Test 2: Delisted security remains
# in the historical universe
assert "BETA" in get_tickers("2015-06-30")

# Test 3: Future IPO must not appear
assert "GAMMA" not in get_tickers("2015-06-30")

# Test 4: ETF must be excluded
assert "DELTA" not in get_tickers("2021-06-30")

# Test 5: OTC security must be excluded
assert "EPSILON" not in get_tickers("2021-06-30")

# Test 6: Listing-date boundary
assert "BETA" not in get_tickers("2011-05-31")
assert "BETA" in get_tickers("2011-06-01")

# Test 7: Delisting-date boundary
assert "BETA" in get_tickers("2018-09-28")
assert "BETA" not in get_tickers("2018-10-01")

print("All 7 PIT universe tests passed.")

All 7 PIT universe tests passed.


In [5]:
# Day 4 — Step 3.2
# Deliberately introduce survivorship bias

# Incorrect approach:
# Remove every security with a delisting date
survivors_only = security_master.loc[
    security_master["delisting_date"].isna()
].copy()

correct_universe = get_pit_universe(
    security_master, "2015-06-30"
)

biased_universe = get_pit_universe(
    survivors_only, "2015-06-30"
)

print(
    "Correct 2015 universe:",
    correct_universe["ticker"].tolist()
)

print(
    "Biased 2015 universe:",
    biased_universe["ticker"].tolist()
)

missing = (
    set(correct_universe["ticker"])
    - set(biased_universe["ticker"])
)

print("Historical securities lost:", missing)

assert "BETA" in missing

print("Survivorship-bias test demonstrated.")

Correct 2015 universe: ['ALPHA', 'BETA']
Biased 2015 universe: ['ALPHA']
Historical securities lost: {'BETA'}
Survivorship-bias test demonstrated.
